#   Modelo e Algoritmo: Plackett–Luce Bayesiano e Power EP

Este notebook apresenta a fundamentação teórica dos modelos e algoritmos utilizados no projeto, com exemplos de código ilustrativos.

---

## 1. Modelo Plackett–Luce

O modelo **Plackett–Luce** é amplamente utilizado para modelar dados de **ranking**. A ideia central é associar a cada item $i$ (e.g., um piloto de F1) um parâmetro positivo de habilidade $w_i > 0$.

### Probabilidade de um ranking

Dado um ranking completo $(\sigma_1, \sigma_2, \ldots, \sigma_n)$, a probabilidade é:

$$P(\sigma \mid \mathbf{w}) = \prod_{k=1}^{n} \frac{w_{\sigma_k}}{\sum_{j=k}^{n} w_{\sigma_j}}$$

**Interpretação:** na posição $k$, o piloto $\sigma_k$ é escolhido entre os que ainda não foram posicionados, com probabilidade proporcional à sua habilidade $w_{\sigma_k}$.

**Exemplo:** Se VER, NOR e LEC têm habilidades $w = (3, 2, 1)$, a probabilidade do ranking VER > NOR > LEC é:

$$P = \frac{3}{6} \cdot \frac{2}{3} \cdot 1 = \frac{1}{3}$$


In [1]:
import numpy as np
from itertools import permutations

def plackett_luce_prob(ranking, skills):
    """Calcula P(ranking | skills) pelo modelo Plackett-Luce."""
    prob = 1.0
    remaining = list(ranking)
    for item in ranking:
        total = sum(skills[i] for i in remaining)
        prob *= skills[item] / total
        remaining.remove(item)
    return prob

pilotos = ['VER', 'NOR', 'LEC']
skills = {'VER': 3, 'NOR': 2, 'LEC': 1}

print('Probabilidades de cada ranking possivel (Plackett-Luce):')
for perm in permutations(pilotos):
    p = plackett_luce_prob(perm, skills)
    print(f'  {" > ".join(perm)}: {p:.4f}')

Probabilidades de cada ranking possivel (Plackett-Luce):
  VER > NOR > LEC: 0.3333
  VER > LEC > NOR: 0.1667
  NOR > VER > LEC: 0.2500
  NOR > LEC > VER: 0.0833
  LEC > VER > NOR: 0.1000
  LEC > NOR > VER: 0.0667


---

## 2. Inferência Bayesiana

Na abordagem **Bayesiana**, os parâmetros de habilidade $\mathbf{w}$ são tratados como **variáveis aleatórias** com distribuições a priori.

$$p(\mathbf{w} \mid \text{dados}) \propto p(\text{dados} \mid \mathbf{w}) \cdot p(\mathbf{w})$$

- $p(\mathbf{w})$: distribuição **a priori** (e.g., Log-Normal ou Gama)
- $p(\text{dados} \mid \mathbf{w})$: **verossimilhança** do modelo Plackett–Luce
- $p(\mathbf{w} \mid \text{dados})$: distribuição **a posteriori** — o que queremos estimar

### Vantagens da abordagem Bayesiana
- Quantifica **incerteza** sobre as habilidades dos pilotos
- Incorpora **conhecimento prévio** (desempenho histórico)
- Permite **atualização sequencial** corrida a corrida


---

## 3. MCMC — Markov Chain Monte Carlo

O **MCMC** amostra distribuições a posteriori complexas construindo uma cadeia de Markov com distribuição estacionária igual à distribuição desejada.

### Algoritmo Metropolis–Hastings

1. Inicialize $\mathbf{w}^{(0)}$
2. Para cada iteração $t$:
   - Proponha $\mathbf{w}^* \sim q(\mathbf{w}^* \mid \mathbf{w}^{(t-1)})$
   - Calcule a razão de aceitação $\alpha$ e aceite/rejeite a proposta

### Comparação: MCMC vs. Power EP

| Critério | MCMC | Power EP |
|----------|------|----------|
| **Exatidão** | Assintoticamente exato | Aproximado |
| **Velocidade** | Lento | Rápido |
| **Escalabilidade** | Limitada | Alta |
| **Convergência** | Garantida | Nem sempre |


In [2]:
import numpy as np

def log_prior(w, alpha=1.0, beta=1.0):
    """Log-prior Gama(alpha, beta) para cada habilidade w_i > 0."""
    if np.any(w <= 0):
        return -np.inf
    return np.sum((alpha - 1) * np.log(w) - beta * w)

def log_likelihood_pl(rankings, w):
    """Log-verossimilhança Plackett-Luce para uma lista de rankings."""
    ll = 0.0
    for ranking in rankings:
        remaining = list(ranking)
        for item in ranking:
            ll += np.log(w[item]) - np.log(sum(w[i] for i in remaining))
            remaining.remove(item)
    return ll

def mcmc_plackett_luce(rankings, items, n_iter=3000, proposal_std=0.2):
    """Random Walk Metropolis-Hastings para o modelo Plackett-Luce."""
    w = {item: 1.0 for item in items}
    samples, accepted = [], 0
    for _ in range(n_iter):
        w_new = {item: w[item] * np.exp(np.random.normal(0, proposal_std)) for item in items}
        log_r = (log_prior(np.array(list(w_new.values()))) + log_likelihood_pl(rankings, w_new)
               - log_prior(np.array(list(w.values()))) - log_likelihood_pl(rankings, w))
        if np.log(np.random.rand()) < log_r:
            w = w_new
            accepted += 1
        samples.append(dict(w))
    print(f'Taxa de aceitacao: {accepted / n_iter:.2%}')
    return samples

# Simulacao: 5 pilotos, 20 corridas
pilotos = ['VER', 'NOR', 'LEC', 'HAM', 'RUS']
true_w = {'VER': 5, 'NOR': 4, 'LEC': 3, 'HAM': 2, 'RUS': 1}

def sample_pl(items, w):
    remaining = items.copy()
    ranking = []
    while remaining:
        weights = np.array([w[i] for i in remaining])
        chosen = np.random.choice(remaining, p=weights / weights.sum())
        ranking.append(chosen)
        remaining.remove(chosen)
    return ranking

np.random.seed(42)
rankings_obs = [sample_pl(pilotos, true_w) for _ in range(20)]
print('Exemplos de rankings observados:')
for r in rankings_obs[:3]:
    print(' > '.join(r))

samples = mcmc_plackett_luce(rankings_obs, pilotos)

burn_in = 1000
posterior_means = {p: np.mean([s[p] for s in samples[burn_in:]]) for p in pilotos}
total_w = sum(posterior_means.values())
print('\nHabilidades estimadas (normalizadas) vs. reais:')
for p in sorted(pilotos, key=lambda x: -posterior_means[x]):
    print(f'  {p}: estimado={posterior_means[p]/total_w:.3f}  real={true_w[p]/sum(true_w.values()):.3f}')

Exemplos de rankings observados:
NOR > RUS > LEC > VER > HAM
VER > NOR > RUS > HAM > LEC
VER > RUS > HAM > NOR > LEC
Taxa de aceitacao: 52.27%

Habilidades estimadas (normalizadas) vs. reais:
  VER: estimado=0.383  real=0.333
  NOR: estimado=0.230  real=0.267
  LEC: estimado=0.194  real=0.200
  HAM: estimado=0.144  real=0.133
  RUS: estimado=0.049  real=0.067


---

## 4. Algoritmo Power EP

O **Power EP** é uma extensão do Expectation Propagation que utiliza **α-divergências**:

$$D_\alpha(p \| q) = \frac{1}{\alpha(1-\alpha)} \left(1 - \int p(x)^\alpha q(x)^{1-\alpha} dx \right)$$

O parâmetro $\alpha$ controla o comportamento da aproximação entre mode-seeking ($\alpha \to 0$) e mean-matching ($\alpha \to 1$, EP padrão). É útil quando o MCMC é computacionalmente inviável — como em temporadas completas com muitos pilotos.

---

## 5. Clustering de Rankings Parciais — Modelo de Mallows

O **Modelo de Mallows** distribui rankings em torno de um consenso $\rho$ com dispersão $\alpha$:

$$P(\sigma \mid \rho, \alpha) \propto \exp(-\alpha \cdot d(\sigma, \rho))$$

onde $d(\sigma, \rho)$ é a **distância de Kendall** (número de inversões). O algoritmo MCMC para clustering (Algoritmo 4 do artigo de referência) agrupa corridas em clusters por tipo de pista (rápidas vs. técnicas). Veja a implementação completa em:
- `src/MCMC_ClusteringPartialRankings.py`
- `04_MCMC_ClusteringPartialRankings.ipynb`

---

## 6. Conclusão

| Componente | Papel no Projeto |
|------------|----------------|
| **Plackett–Luce** | Modela a probabilidade de um ranking dado as habilidades dos pilotos |
| **Bayes Hierárquico** | Adiciona priors e quantifica incerteza |
| **MCMC** | Amostra a posteriori de forma exata (assintoticamente) |
| **Power EP** | Alternativa aproximada e escalável ao MCMC |
| **Modelo de Mallows** | Clustering de corridas por perfil de pista |
